# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [4]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [8]:
# Calculate revenue for each order
df['revenue'] = df['qty'] * df['price']

revenue = df['revenue'].sum()
units = df['qty'].sum()

print(f"Revenue: ${revenue}")
print(f"Units Sold: {units}")

df

# This just calculates the revenue by multiplying quantity with price.
# Across all 400 orders, total revenue reached $8520 with 783 total units sold.

Revenue: $8520.0
Units Sold: 783


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5
...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0
396,V-01,Merch,2,24.0,48.0
397,V-10,Food,3,7.5,22.5
398,V-18,Merch,2,24.0,48.0


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [15]:
by_category = df.groupby('category').agg(
    revenue=('revenue', 'sum'),
    units=('qty', 'sum')
).reset_index()

by_category['share_pct'] = (by_category['revenue'] / revenue) * 100
by_category = by_category.sort_values(by='revenue', ascending=False)

by_category

# This calculates the total revenue and units sold for each category, then computes each category's share of all revenue in this dataset.

,category,revenue,units,share_pct
1,Food,4293.0,362,50.387324
2,Merch,1771.5,158,20.792254
0,Drink,1554.0,178,18.239437
3,RainGear,901.5,85,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [16]:
vendor_summary = df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).sort_values(by='avg_order_revenue', ascending=False)

vendor_summary

# Looks like it's V-01 that has the highest average order revenue, so they're getting more revenue per transaction.

,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [25]:
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = (merch_revenue / revenue) * 100

print(f"Merch Revenue Share: {merch_share:.1f}%")

# This calculates the total revenue from the 'Merch' category and then computes its share of the overall revenue.

Merch Revenue Share: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [22]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df)
assert np.isclose(joined['revenue'].sum(), revenue)

joined[joined['vendor_name'].isna()]['vendor_id'].unique()

<StringArray>
['V-18']
Length: 1, dtype: str

**The unmatched vendor, and what I did about it:** _V-18 is missing from the vendor names, so a left join is used to retain everything else and leave the vender name for those orders as null._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [23]:
pivot_report = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)

pivot_report

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [24]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

Vendors should prioritize selling Food and Merch since those accounted for most of the sales ($4293 and $1771.50 respectively)

Q5 is the least trustworthy considering we just left out an entire vendor so we cannot do reliable vendor-specific analysis